# Multiscale MD Pipeline: DFT → LAMMPS → AIMD → Analysis

This notebook demonstrates the **complete multiscale simulation pipeline** in SHALOM,
from DFT input generation through classical MD (LAMMPS) to ab-initio MD (VASP AIMD)
and full trajectory analysis with publication-quality figures.

## What We Will Do

| Step | Task | Backend | Time |
|------|------|---------|------|
| 1 | Crystal structure preparation | ASE | < 1 sec |
| 2 | Symmetry analysis | spglib | < 1 sec |
| 3 | XRD pattern | pymatgen | < 1 sec |
| 4 | DFT input generation (VASP relax + QE SCF) | VASP, QE | < 5 sec |
| 5 | LAMMPS MD inputs — 3 materials (Fe, Si, Ar) | LAMMPS | < 5 sec |
| 6 | VASP AIMD input generation | VASP | < 1 sec |
| 7 | Synthetic MD trajectory (Fe BCC 300 K) | numpy | < 5 sec |
| 8 | MD analysis (RDF, MSD, VACF, diffusion) | SHALOM | < 5 sec |
| 9 | Publication-quality MD plots (5 figures) | matplotlib | < 5 sec |
| 10 | Summary | — | < 1 sec |

**Total: ~5 minutes** (no external software required)

## Key Features Demonstrated

- **Triple backend support**: VASP, QE, LAMMPS input generation
- **Material-aware auto-detection**: Force field selection (EAM/Tersoff/LJ) from element composition
- **VASP AIMD**: Nosé-Hoover NVT molecular dynamics setup
- **MD trajectory analysis**: RDF, MSD, VACF, diffusion coefficient, equilibration detection
- **5 publication-quality plots**: Energy, Temperature, MSD, RDF, VACF

In [ ]:
import os
import warnings

import numpy as np

%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.dpi"] = 150
import matplotlib.pyplot as plt

from ase.build import bulk

warnings.filterwarnings("ignore", category=DeprecationWarning)

# === Output directory ===
OUTPUT_ROOT = os.path.expanduser("~/Desktop/shalom-tutorials/fe_md_study")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print(f"Output directory: {OUTPUT_ROOT}")
print(f"NumPy version:    {np.__version__}")

## 1. Structure Preparation

Iron crystallizes in the BCC structure (space group Im̅3m, #229) at room temperature.
We create a conventional cell (2 atoms) and a 3×3×3 supercell (54 atoms) for MD simulations.

In [ ]:
# BCC Fe — conventional cell (2 atoms) for proper supercell construction
fe = bulk("Fe", "bcc", a=2.87, cubic=True)

print("=== Fe BCC Conventional Cell ===")
print(f"Formula:     {fe.get_chemical_formula()}")
print(f"Atoms:       {len(fe)}")
print(f"Cell (\u00c5):")
for i, v in enumerate(fe.cell):
    print(f"  a{i+1} = [{v[0]:8.4f}, {v[1]:8.4f}, {v[2]:8.4f}]")
print(f"Volume:      {fe.get_volume():.2f} \u00c5\u00b3")

# 3x3x3 supercell for MD (54 atoms)
fe_super = fe * (3, 3, 3)
print(f"\n=== 3\u00d73\u00d73 Supercell ===")
print(f"Atoms:       {len(fe_super)}")
print(f"Cell (\u00c5):")
for i, v in enumerate(fe_super.cell):
    print(f"  a{i+1} = [{v[0]:8.4f}, {v[1]:8.4f}, {v[2]:8.4f}]")
print(f"Volume:      {fe_super.get_volume():.2f} \u00c5\u00b3")
print(f"\nNearest-neighbor distance: {2.87 * 3**0.5 / 2:.3f} \u00c5 (a\u221a3/2)")

## 2. Crystal Symmetry Analysis

Before any calculations, we characterize the crystal symmetry using spglib.
BCC Fe should be space group Im̅3m (#229) with point group m̅3m.

In [ ]:
from shalom.analysis import analyze_symmetry, is_spglib_available

if is_spglib_available():
    sym = analyze_symmetry(fe)
    print("=== Crystal Symmetry ===")
    print(f"Space group:    {sym.space_group_symbol} (#{sym.space_group_number})")
    print(f"Crystal system: {sym.crystal_system}")
    print(f"Point group:    {sym.point_group}")
    print(f"Lattice type:   {sym.lattice_type}")
    print(f"Hall symbol:    {sym.hall_symbol}")
    print(f"Is primitive:   {sym.is_primitive}")
    print(f"Symmetry ops:   {sym.n_operations}")
    print(f"Wyckoff:        {sym.wyckoff_letters}")
else:
    print("spglib not installed. Run: pip install spglib")

## 3. X-ray Diffraction Pattern

Simulated powder XRD pattern using Cu K\u03b1 radiation (\u03bb = 1.5406 \u00c5).
BCC Fe shows characteristic reflections: (110), (200), (211), (220), etc.

In [ ]:
from shalom.analysis import calculate_xrd, is_xrd_available
from shalom.plotting import XRDPlotter

if is_xrd_available():
    xrd_result = calculate_xrd(fe, wavelength="CuKa", two_theta_range=(20, 120))

    print(f"Number of peaks: {xrd_result.n_peaks}")
    print(f"\nTop peaks:")
    print(f"{'2\u03b8 (deg)':>12}  {'Intensity':>10}  {'(h k l)':>10}  {'d (\u00c5)':>8}")
    print("-" * 50)
    order = np.argsort(xrd_result.intensities)[::-1]
    for i in order[:5]:
        hkl = xrd_result.hkl_indices[i]
        print(f"{xrd_result.two_theta[i]:12.2f}  {xrd_result.intensities[i]:10.1f}  "
              f"({hkl[0]:2d} {hkl[1]:2d} {hkl[2]:2d})  {xrd_result.d_spacings[i]:8.4f}")

    xrd_path = os.path.join(OUTPUT_ROOT, "fe_xrd.png")
    fig = XRDPlotter(xrd_result).plot(
        output_path=xrd_path,
        title="Fe (BCC) \u2014 Simulated XRD (Cu K\u03b1)",
    )
    plt.show()
    print(f"\nSaved: {xrd_path}")
else:
    print("pymatgen not installed. Run: pip install pymatgen")

## 4. DFT Input Generation

Generate DFT input files for two backends:
- **VASP**: Variable-cell relaxation (vc-relax)
- **QE**: Self-consistent field (SCF)

These files are ready to submit to the respective DFT codes. No actual DFT execution
is performed here.

In [ ]:
from shalom.direct_run import direct_run, DirectRunConfig

# --- 4a. VASP Relaxation ---
vasp_dir = os.path.join(OUTPUT_ROOT, "02_vasp_relax")
vasp_config = DirectRunConfig(
    backend_name="vasp",
    calc_type="relaxation",
    accuracy="standard",
    output_dir=vasp_dir,
)
vasp_result = direct_run("Fe", vasp_config)

print("=== VASP Relaxation ===")
print(f"Output: {vasp_result.output_dir}")
print(f"Files:  {vasp_result.files_generated}")
if vasp_result.auto_detected:
    print(f"Auto:   {vasp_result.auto_detected}")

# Show INCAR highlights
incar_path = os.path.join(vasp_dir, "INCAR")
if os.path.exists(incar_path):
    print(f"\n--- INCAR (first 20 lines) ---")
    with open(incar_path) as f:
        for i, line in enumerate(f):
            if i >= 20:
                print("...")
                break
            print(line.rstrip())

In [ ]:
# --- 4b. QE SCF ---
qe_dir = os.path.join(OUTPUT_ROOT, "03_qe_scf")
qe_config = DirectRunConfig(
    backend_name="qe",
    calc_type="scf",
    accuracy="standard",
    output_dir=qe_dir,
)
qe_result = direct_run("Fe", qe_config)

print("=== QE SCF ===")
print(f"Output: {qe_result.output_dir}")
print(f"Files:  {qe_result.files_generated}")

# Show pw.in highlights
pwin_path = os.path.join(qe_dir, "pw.in")
if os.path.exists(pwin_path):
    print(f"\n--- pw.in (first 30 lines) ---")
    with open(pwin_path) as f:
        for i, line in enumerate(f):
            if i >= 30:
                print("...")
                break
            print(line.rstrip())

## 5. LAMMPS Input Generation — 3 Materials

SHALOM's LAMMPS backend **automatically detects** the optimal force field based on
element composition:

| Material | Structure | Force Field | Priority |
|----------|-----------|-------------|----------|
| Fe | BCC | EAM/alloy | 90 (metal) |
| Si | Diamond | Tersoff | 85 (covalent) |
| Ar | FCC | LJ/cut | 30 (noble gas) |

No manual `--pair-style` needed — the YAML database (`lammps_potentials.yaml`)
maps element sets to force fields with priority-based selection.

In [ ]:
# --- 5a. Fe (BCC) → EAM/alloy (auto-detected) ---
lammps_fe_dir = os.path.join(OUTPUT_ROOT, "04_lammps_fe")
lammps_fe_config = DirectRunConfig(
    backend_name="lammps",
    accuracy="standard",
    output_dir=lammps_fe_dir,
)
lammps_fe_result = direct_run("Fe", lammps_fe_config)

print("=== LAMMPS: Fe (BCC) ===")
print(f"Output:      {lammps_fe_result.output_dir}")
print(f"Files:       {lammps_fe_result.files_generated}")
print(f"Auto-detect: {lammps_fe_result.auto_detected}")

# Show in.lammps
in_lammps = os.path.join(lammps_fe_dir, "in.lammps")
if os.path.exists(in_lammps):
    print(f"\n--- in.lammps ---")
    with open(in_lammps) as f:
        print(f.read())

In [ ]:
# --- 5b. Si (Diamond) → Tersoff (auto-detected) ---
lammps_si_dir = os.path.join(OUTPUT_ROOT, "05_lammps_si")
lammps_si_config = DirectRunConfig(
    backend_name="lammps",
    accuracy="standard",
    output_dir=lammps_si_dir,
)
lammps_si_result = direct_run("Si", lammps_si_config)

print("=== LAMMPS: Si (Diamond) ===")
print(f"Output:      {lammps_si_result.output_dir}")
print(f"Files:       {lammps_si_result.files_generated}")
print(f"Auto-detect: {lammps_si_result.auto_detected}")

in_lammps = os.path.join(lammps_si_dir, "in.lammps")
if os.path.exists(in_lammps):
    print(f"\n--- in.lammps ---")
    with open(in_lammps) as f:
        print(f.read())

In [ ]:
# --- 5c. Ar (FCC) → Lennard-Jones (auto-detected) ---
lammps_ar_dir = os.path.join(OUTPUT_ROOT, "06_lammps_ar")
lammps_ar_config = DirectRunConfig(
    backend_name="lammps",
    accuracy="standard",
    output_dir=lammps_ar_dir,
)
lammps_ar_result = direct_run("Ar", lammps_ar_config)

print("=== LAMMPS: Ar (FCC) ===")
print(f"Output:      {lammps_ar_result.output_dir}")
print(f"Files:       {lammps_ar_result.files_generated}")
print(f"Auto-detect: {lammps_ar_result.auto_detected}")

in_lammps = os.path.join(lammps_ar_dir, "in.lammps")
if os.path.exists(in_lammps):
    print(f"\n--- in.lammps ---")
    with open(in_lammps) as f:
        print(f.read())

# Show data file for Ar (LJ has pair_coeff inline, no potential file)
data_lammps = os.path.join(lammps_ar_dir, "data.lammps")
if os.path.exists(data_lammps):
    print(f"\n--- data.lammps ---")
    with open(data_lammps) as f:
        print(f.read())

### Force Field Auto-Detection Summary

Notice how SHALOM automatically selected the correct force field for each material:
- **Fe** → `eam/alloy` (metallic bonding, highest priority for transition metals)
- **Si** → `tersoff` (covalent bonding, ideal for Group IV semiconductors)
- **Ar** → `lj/cut 10.0` (van der Waals, appropriate for noble gases)

Each force field also sets appropriate defaults:
- **Timestep**: Fe 1.0 fs, Si 0.5 fs, Ar 5.0 fs
- **Thermostat damping**: Tuned per force field from `lammps_potentials.yaml`

## 6. VASP AIMD Input Generation

VASP ab-initio molecular dynamics (AIMD) uses DFT forces at each timestep.
Key INCAR settings: `IBRION=0` (MD), `ISYM=0` (no symmetry), `SMASS=0`
(Nosé-Hoover thermostat for NVT ensemble).

In [ ]:
aimd_dir = os.path.join(OUTPUT_ROOT, "07_vasp_aimd")
aimd_config = DirectRunConfig(
    backend_name="vasp",
    calc_type="aimd",
    accuracy="standard",
    output_dir=aimd_dir,
)
aimd_result = direct_run("Fe", aimd_config)

print("=== VASP AIMD ===")
print(f"Output: {aimd_result.output_dir}")
print(f"Files:  {aimd_result.files_generated}")

# Verify AIMD-specific INCAR tags
incar_path = os.path.join(aimd_dir, "INCAR")
if os.path.exists(incar_path):
    print(f"\n--- INCAR (AIMD settings) ---")
    with open(incar_path) as f:
        content = f.read()
        print(content)

    # Verify key AIMD tags
    checks = {
        "IBRION = 0": "MD mode",
        "ISYM = 0": "No symmetry",
        "SMASS = 0": "Nos\u00e9-Hoover NVT",
        "POTIM": "Timestep (fs)",
        "TEBEG": "Start temperature (K)",
        "TEEND": "End temperature (K)",
    }
    print("\nAIMD Tag Verification:")
    for tag, desc in checks.items():
        found = tag in content
        status = "\u2713" if found else "\u2717"
        print(f"  {status} {tag:20s} ({desc})")

## 7. Synthetic MD Trajectory

To demonstrate the full analysis pipeline without requiring LAMMPS/VASP execution,
we generate a **physically realistic synthetic trajectory** for Fe BCC at 300 K.

### Physics of the synthetic data:
- **Thermal displacements**: Debye-Waller model, $u_{\mathrm{rms}} = 0.08$ \u00c5 for BCC Fe at 300 K
- **Velocities**: Maxwell-Boltzmann distribution, $v_{\mathrm{rms}} = \sqrt{k_BT/m_{\mathrm{Fe}}}$
- **Temperature**: Canonical fluctuations $\sigma_T = T\sqrt{2/(3N)}$
- **Energy**: Harmonic oscillation around equilibrium with thermal noise
- **Diffusion**: Small random walk component to produce measurable MSD

In [ ]:
from shalom.backends.base import MDTrajectoryData

# Physical parameters
n_frames = 1000
n_atoms = len(fe_super)           # 54 atoms
dt = 1.0                          # fs
T_target = 300.0                  # K
u_rms = 0.08                      # Debye-Waller rms displacement (\u00c5) for Fe at 300 K
m_fe = 55.845                     # Fe atomic mass (amu)

# Constants
kB = 8.617333e-5                  # eV/K
kB_SI = 1.380649e-23             # J/K
amu_to_kg = 1.66054e-27          # kg/amu

rng = np.random.default_rng(seed=42)
cell = np.array(fe_super.cell)    # (3, 3) \u00c5
eq_positions = fe_super.get_positions()  # (54, 3) \u00c5
species = fe_super.get_chemical_symbols()

print(f"Generating synthetic Fe NVT trajectory...")
print(f"  Atoms: {n_atoms}")
print(f"  Frames: {n_frames}")
print(f"  Timestep: {dt} fs")
print(f"  Target T: {T_target} K")
print(f"  u_rms: {u_rms} \u00c5")

In [ ]:
# --- Generate trajectory arrays ---

# Time array
times = np.arange(n_frames) * dt  # fs

# Positions: equilibrium + thermal displacement + small diffusive drift
positions = np.zeros((n_frames, n_atoms, 3))
drift = np.zeros((n_atoms, 3))
drift_step = 0.001  # \u00c5 per frame (very slow diffusion, solid state)

for t in range(n_frames):
    thermal = rng.normal(0.0, u_rms, size=(n_atoms, 3))
    drift += rng.normal(0.0, drift_step, size=(n_atoms, 3))
    positions[t] = eq_positions + thermal + drift

# Velocities: Maxwell-Boltzmann at T_target
# v_rms = sqrt(kB*T / m) in \u00c5/fs
v_rms = np.sqrt(kB_SI * T_target / (m_fe * amu_to_kg)) * 1e-5  # m/s -> \u00c5/fs
velocities = rng.normal(0.0, v_rms, size=(n_frames, n_atoms, 3))

# Temperature: canonical fluctuations
sigma_T = T_target * np.sqrt(2.0 / (3.0 * n_atoms))
temperatures = rng.normal(T_target, sigma_T, size=n_frames)
temperatures = np.clip(temperatures, 200, 400)  # physical bounds

# Energy: equilibrium + thermal fluctuations
E_per_atom = -4.28  # eV/atom (approximate for Fe BCC)
E_eq = E_per_atom * n_atoms
sigma_E = n_atoms * kB * T_target  # canonical energy fluctuation
energies = rng.normal(E_eq, sigma_E, size=n_frames)

# Kinetic energies from temperature: KE = (3/2) N kB T
kinetic_energies = 1.5 * n_atoms * kB * temperatures
potential_energies = energies - kinetic_energies

# Build MDTrajectoryData
traj = MDTrajectoryData(
    positions=positions,
    energies=energies,
    temperatures=temperatures,
    times=times,
    species=species,
    cell_vectors=cell,
    velocities=velocities,
    kinetic_energies=kinetic_energies,
    potential_energies=potential_energies,
    ensemble="NVT",
    timestep_fs=dt,
    source="synthetic",
)

print(f"\n=== MDTrajectoryData ===")
print(f"n_frames:   {traj.n_frames}")
print(f"n_atoms:    {traj.n_atoms}")
print(f"Ensemble:   {traj.ensemble}")
print(f"Source:     {traj.source}")
print(f"T (mean):   {temperatures.mean():.1f} \u00b1 {temperatures.std():.1f} K")
print(f"E (mean):   {energies.mean():.3f} \u00b1 {energies.std():.3f} eV")
print(f"v_rms:      {v_rms:.6f} \u00c5/fs ({v_rms * 1e5:.1f} m/s)")

## 8. MD Trajectory Analysis

SHALOM provides a complete MD analysis toolkit:

| Function | What it computes |
|----------|------------------|
| `analyze_md_trajectory()` | Full analysis (all below) → `MDResult` |
| `compute_rdf()` | Radial distribution function g(r) |
| `compute_msd()` | Mean square displacement |
| `compute_vacf()` | Velocity autocorrelation function |
| `compute_diffusion_coefficient()` | D from Einstein relation |
| `detect_equilibration()` | Equilibration frame detection |

In [ ]:
from shalom.analysis.md import (
    analyze_md_trajectory,
    compute_rdf,
    compute_msd,
    compute_vacf,
    compute_diffusion_coefficient,
    detect_equilibration,
)

# Full analysis in one call
md_result = analyze_md_trajectory(
    traj,
    r_max=8.0,             # \u00c5 (half the box size)
    n_rdf_bins=200,
    compute_vacf_flag=True,
    max_vacf_lag=200,
)

print("=== MD Analysis Results ===")
print(f"\nThermodynamics (post-equilibration):")
print(f"  Avg temperature:  {md_result.avg_temperature:.1f} \u00b1 {md_result.temperature_std:.1f} K")
print(f"  Avg energy:       {md_result.avg_energy:.3f} eV")
print(f"  Equilibrated:     {md_result.is_equilibrated}")
print(f"  Equil. frame:     {md_result.equilibration_step}")

print(f"\nDiffusion:")
if md_result.diffusion_coefficient is not None:
    print(f"  D = {md_result.diffusion_coefficient:.4e} cm\u00b2/s")

print(f"\nRDF:")
if md_result.rdf_r is not None:
    # Skip r < 1.5 \u00c5 to avoid normalization artifact at r\u21920
    mask = md_result.rdf_r > 1.5
    peak_idx = np.where(mask)[0][np.argmax(md_result.rdf_g[mask])]
    print(f"  First peak at r = {md_result.rdf_r[peak_idx]:.2f} \u00c5 (g = {md_result.rdf_g[peak_idx]:.2f})")
    nn_fe = 2.87 * np.sqrt(3) / 2
    print(f"  Expected (BCC Fe nn): {nn_fe:.3f} \u00c5")
    print(f"  Pairs: {md_result.rdf_pairs}")

print(f"\nVACF:")
if md_result.vacf is not None:
    print(f"  Points: {len(md_result.vacf)}")
    print(f"  VACF(0) = {md_result.vacf[0]:.4f} (should be 1.0)")

In [ ]:
# Individual function calls (for demonstration)

# RDF
r, g_r = compute_rdf(traj, r_max=8.0, n_bins=200)
print(f"RDF: {len(r)} bins, r_max = {r[-1]:.2f} \u00c5")

# MSD
msd_t, msd = compute_msd(traj)
print(f"MSD: {len(msd_t)} points, final MSD = {msd[-1]:.4f} \u00c5\u00b2")

# VACF
vacf_t, vacf = compute_vacf(traj, max_lag=200)
print(f"VACF: {len(vacf_t)} points")

# Diffusion coefficient from MSD
D = compute_diffusion_coefficient(msd_t, msd)
print(f"Diffusion coefficient: D = {D:.4e} cm\u00b2/s")

# Equilibration detection
eq_frame, is_eq = detect_equilibration(energies)
print(f"Equilibration: frame {eq_frame}, equilibrated = {is_eq}")

## 9. MD Plots — Publication-Quality Figures

Five figures covering all aspects of the MD trajectory:

1. **Energy vs Time** — total, kinetic, potential
2. **Temperature vs Time** — instantaneous + running average + target
3. **MSD vs Time** — with Einstein fit for diffusion
4. **RDF g(r)** — radial distribution function
5. **VACF** — velocity autocorrelation function

In [ ]:
from shalom.plotting import MDEnergyPlotter, MDTemperaturePlotter, MSDPlotter, RDFPlotter

analysis_dir = os.path.join(OUTPUT_ROOT, "08_analysis")
os.makedirs(analysis_dir, exist_ok=True)

# 1. Energy vs Time
energy_path = os.path.join(analysis_dir, "fe_md_energy.png")
fig = MDEnergyPlotter(traj).plot(
    output_path=energy_path,
    title="Fe BCC (300 K, NVT) \u2014 Energy vs Time",
    show_components=True,
)
plt.show()
print(f"Saved: {energy_path}")

# 2. Temperature vs Time
temp_path = os.path.join(analysis_dir, "fe_md_temperature.png")
fig = MDTemperaturePlotter(traj).plot(
    output_path=temp_path,
    title="Fe BCC (300 K, NVT) \u2014 Temperature vs Time",
    target_temperature=300.0,
    running_avg_window=50,
)
plt.show()
print(f"Saved: {temp_path}")

In [ ]:
# 3. MSD vs Time
msd_path = os.path.join(analysis_dir, "fe_md_msd.png")
fig = MSDPlotter(md_result).plot(
    output_path=msd_path,
    title="Fe BCC (300 K) \u2014 Mean Square Displacement",
    show_fit=True,
)
plt.show()
print(f"Saved: {msd_path}")

# 4. RDF g(r)
rdf_path = os.path.join(analysis_dir, "fe_md_rdf.png")
fig = RDFPlotter(md_result).plot(
    output_path=rdf_path,
    title="Fe BCC (300 K) \u2014 Radial Distribution Function",
)
plt.show()
print(f"Saved: {rdf_path}")

In [ ]:
# 5. VACF (custom matplotlib plot)
if md_result.vacf is not None and len(md_result.vacf) > 0:
    fig, ax = plt.subplots(figsize=(7, 5), dpi=150)
    ax.plot(md_result.vacf_t, md_result.vacf, color="royalblue", lw=1.5)
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    ax.set_xlabel("Time lag (fs)")
    ax.set_ylabel("VACF (normalized)")
    ax.set_title("Fe BCC (300 K) \u2014 Velocity Autocorrelation Function")
    ax.set_xlim(0, md_result.vacf_t[-1])
    fig.tight_layout()

    vacf_path = os.path.join(analysis_dir, "fe_md_vacf.png")
    fig.savefig(vacf_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {vacf_path}")
else:
    print("VACF not available (no velocity data).")

## 10. Summary

Complete results from the multiscale MD pipeline.

In [ ]:
print("=" * 70)
print("Fe BCC Multiscale MD Pipeline \u2014 Summary")
print("=" * 70)

# Structure
print(f"\n\u250c\u2500 Structure")
print(f"\u2502  Formula:        Fe (BCC)")
print(f"\u2502  Lattice:        a = 2.87 \u00c5")
print(f"\u2502  Supercell:      3\u00d73\u00d73 = {n_atoms} atoms")
if 'sym' in dir():
    print(f"\u2502  Space group:    {sym.space_group_symbol} (#{sym.space_group_number})")
    print(f"\u2502  Point group:    {sym.point_group}")

# Input generation
print(f"\u2502")
print(f"\u251c\u2500 Input Generation")
print(f"\u2502  VASP relax:     {vasp_dir}")
print(f"\u2502  QE SCF:         {qe_dir}")
print(f"\u2502  LAMMPS Fe(EAM): {lammps_fe_dir}")
print(f"\u2502  LAMMPS Si(Ter): {lammps_si_dir}")
print(f"\u2502  LAMMPS Ar(LJ):  {lammps_ar_dir}")
print(f"\u2502  VASP AIMD:      {aimd_dir}")

# MD Analysis
print(f"\u2502")
print(f"\u251c\u2500 MD Analysis (Synthetic Fe 300 K NVT)")
print(f"\u2502  Temperature:    {md_result.avg_temperature:.1f} \u00b1 {md_result.temperature_std:.1f} K")
print(f"\u2502  Avg energy:     {md_result.avg_energy:.3f} eV")
if md_result.rdf_r is not None:
    mask = md_result.rdf_r > 1.5
    peak_idx = np.where(mask)[0][np.argmax(md_result.rdf_g[mask])]
    print(f"\u2502  RDF 1st peak:   r = {md_result.rdf_r[peak_idx]:.2f} \u00c5")
if md_result.diffusion_coefficient is not None:
    print(f"\u2502  Diffusion:      D = {md_result.diffusion_coefficient:.4e} cm\u00b2/s")
print(f"\u2502  Equilibrated:   {md_result.is_equilibrated}")

# Output files
print(f"\u2502")
print(f"\u2514\u2500 Output Files")
for root, dirs, files in os.walk(OUTPUT_ROOT):
    level = root.replace(OUTPUT_ROOT, "").count(os.sep)
    indent = "   " * level
    dirname = os.path.basename(root)
    if level == 0:
        dirname = OUTPUT_ROOT
    print(f"   {indent}{dirname}/")
    for f in sorted(files):
        print(f"   {indent}  {f}")

print(f"\nAll outputs saved to: {OUTPUT_ROOT}")
print(f"\nPipeline complete!")